# Week 7 — Model Evaluation, Validation & Metrics
## Notebook 2: Holdout Validation — Designing Your Data Splits

| | |
|---|---|
| **Difficulty** | ⭐⭐ (Beginner–Intermediate) |
| **Estimated Time** | 2 hours |
| **Theme** | Credit Card Fraud Detection |
| **Prerequisites** | Notebook 1 (Generalization), basic sklearn |

---

## The Exam Analogy — and Why It's Sacred

Imagine you're a teacher designing a certification exam for fraud analysts:

- **Study material (textbook)** → Your **training set**. The student can read this as many times as needed, highlight it, memorize it.
- **Practice tests** → Your **validation set**. The student can take these repeatedly, see their score, and go back and study more.
- **Final exam** → Your **test set**. One chance only. No retakes. No "let me study the questions first."

Now imagine a student sneaks a look at the final exam before taking it. They score 98%. Is that a valid certification? **No — the score is meaningless.** The exam has been contaminated.

**This is exactly what happens when you use the test set during model development.**

Every time you look at test set performance and then adjust your model, retrain, or add features — the test set is no longer a genuine measure of generalization. It has leaked into your development process. The score you report is optimistically biased.

---

## What You Will Learn

1. Why you need separate train, validation, and test sets
2. The test set is a one-time instrument — treat it like a final exam
3. How to choose split ratios based on dataset size
4. Stratification: preserving class balance in imbalanced fraud data
5. Random seeds, reproducibility, and why they matter for fair comparisons
6. Common mistakes that silently inflate your reported accuracy

---
## Part 1: Why Separate Sets at All?

### The Core Problem

When a model is trained on data, it partially *memorizes* that data. Even a well-regularized model is biased toward patterns it has seen. Evaluating it on the same data it trained on tells you how well it memorized — not how well it generalizes.

### Two-Way vs. Three-Way Split

| Setup | When to Use | Risk |
|---|---|---|
| **Train / Test only** | Fixed model, no hyperparameter tuning | Low — single evaluation |
| **Train / Val / Test** | Standard ML: tune hyperparameters, compare models | Medium — val set guides decisions |
| **Cross-Validation** | Small datasets, need reliable val estimate | Most robust, highest compute |

### The Three Sets and Their Roles

**Training set:**  
- Used to fit model parameters (weights, splits, coefficients)
- Model *sees* this data → always biased toward it
- Can be used as many times as needed

**Validation set:**  
- Used to tune hyperparameters (depth, C, k, learning rate)
- Used to choose between model architectures
- Can be evaluated multiple times (but each use slightly contaminates it)
- **NOT** used to fit model parameters

**Test set:**  
- Final unbiased estimate of generalization performance
- Touched **exactly once**, at the very end, after all decisions are made
- If you change the model after seeing the test score → it is no longer valid
- Report this number as your model's performance

In [ ]:
# ── Setup: Imports and consistent styling ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
np.random.seed(42)

print("Libraries loaded.")
print("Today's lesson: designing data splits that give honest performance estimates.")

In [ ]:
# ── Create a realistic credit card fraud dataset ──────────────────────────────
# 10,000 transactions | 10 features | 2% fraud rate
# This represents one quarter of transactions for a mid-sized bank

X, y = make_classification(
    n_samples=10_000,
    n_features=10,
    n_informative=6,       # 6 features have real signal
    n_redundant=2,         # 2 are derived from informative features
    n_clusters_per_class=2,# fraud types cluster (card skimming vs account takeover)
    weights=[0.98, 0.02],  # 2% fraud — realistic imbalance
    flip_y=0.005,          # 0.5% label noise
    random_state=42
)

feature_names = [
    'txn_amount', 'time_of_day', 'merchant_category', 'distance_from_home',
    'foreign_flag', 'txn_velocity_1h', 'avg_txn_amt_30d',
    'card_age_days', 'noise_1', 'noise_2'
]

df = pd.DataFrame(X, columns=feature_names)
df['is_fraud'] = y

total_fraud = y.sum()
print(f"Dataset: {len(df):,} transactions")
print(f"Fraud cases: {total_fraud} ({total_fraud/len(df)*100:.2f}%)")
print(f"Legitimate: {(y==0).sum():,} ({(y==0).sum()/len(df)*100:.2f}%)")
print()
print("Class imbalance is a critical challenge for holdout splitting.")
print("A naive split could put ALL fraud cases in train and NONE in test.")

---
## Part 2: Stratification — Never Skip This for Imbalanced Data

### The Problem With Random Splits

With 2% fraud and a 20% test split, the test set should have roughly **0.02 × 2000 = 40 fraud cases**. But in a random split, you might get 28, or 55, or — in a very unlucky draw — 0.

**Consequences of unstratified splits:**
- Test set with very few fraud cases → wildly variable performance estimates
- Test set with 0 fraud cases → model appears 100% accurate (predicts all legitimate)
- Different runs give different numbers → you can't compare models fairly

**Solution: `stratify=y`** — sklearn preserves the class proportions from the original dataset in each split.

$$\text{Fraud rate in split} \approx \text{Fraud rate in full dataset}$$

In [ ]:
# ── Stratification Demo: With vs. Without ─────────────────────────────────────
# Run 50 different splits with and without stratify
# Measure how much the fraud rate varies across splits
# A stable fraud rate = reliable, comparable evaluation

n_trials = 50
unstrat_rates = []  # fraud rate in test set WITHOUT stratify
strat_rates   = []  # fraud rate in test set WITH stratify

for seed in range(n_trials):
    # Without stratification
    _, X_te_u, _, y_te_u = train_test_split(
        X, y, test_size=0.20, random_state=seed   # no stratify
    )
    unstrat_rates.append(y_te_u.mean())

    # With stratification
    _, X_te_s, _, y_te_s = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y
    )
    strat_rates.append(y_te_s.mean())

# ── Plot comparison ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, rates, title, color in [
    (axes[0], unstrat_rates, 'WITHOUT Stratify\n(Random Split)', 'tomato'),
    (axes[1], strat_rates,   'WITH Stratify\n(Stratified Split)', 'steelblue')
]:
    ax.hist(rates, bins=15, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(y.mean(), color='black', linestyle='--', lw=2,
               label=f'True rate: {y.mean():.3f}')
    ax.set_xlabel('Fraud Rate in Test Set', fontsize=11)
    ax.set_ylabel('Count (across 50 random splits)', fontsize=11)
    ax.set_title(f'Fraud Rate in Test Set: {title}\n'
                 f'Std Dev: {np.std(rates):.4f}', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Stratification Stabilizes Fraud Rate Across Splits\n'
             'Wider spread = less reliable evaluation', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"Without stratify — Std Dev of fraud rate across splits: {np.std(unstrat_rates):.4f}")
print(f"With stratify   — Std Dev of fraud rate across splits: {np.std(strat_rates):.6f}")
print(f"Stratification reduces variance by {np.std(unstrat_rates)/np.std(strat_rates):.0f}x")

---
## Part 3: The Three-Way Split in Practice

### Standard Split: 70% Train / 15% Validation / 15% Test

sklearn's `train_test_split` only does two-way splits. To do a three-way split, we apply it **twice**:
1. Split off the test set first (15%)
2. Split the remaining 85% into train (70/85 ≈ 82.4%) and val (15/85 ≈ 17.6%)

**Why test set first?** This ensures the test set is never touched again. It's locked away from the moment we create it.

$$\underbrace{70\%}_{\text{Train}} + \underbrace{15\%}_{\text{Val}} + \underbrace{15\%}_{\text{Test}} = 100\%$$

In [ ]:
# ── Three-Way Split: 70/15/15 with stratification ─────────────────────────────
# Step 1: carve out test set (15%) — locked away, never used until the very end
# Step 2: split remaining data into train (70%) and validation (15%)

# Step 1: Separate test set (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y          # preserve 2% fraud rate in both halves
)

# Step 2: Split remaining 85% into train (70% of total) and val (15% of total)
# val_size = 15/85 ≈ 0.1765 of the trainval pool
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.1765,   # 15% of total = 17.65% of the 85% trainval pool
    random_state=42,
    stratify=y_trainval
)

# Verify sizes and fraud rates
splits = {
    'Train':      (X_train, y_train),
    'Validation': (X_val,   y_val),
    'Test':       (X_test,  y_test),
}

print("Three-Way Split Summary")
print("=" * 55)
print(f"{'Split':<15} {'Size':>8} {'% of Total':>12} {'Fraud Rate':>12}")
print("-" * 55)
for name, (Xs, ys) in splits.items():
    pct = len(Xs) / len(X) * 100
    fraud_rate = ys.mean() * 100
    print(f"{name:<15} {len(Xs):>8,} {pct:>11.1f}% {fraud_rate:>11.2f}%")
print("-" * 55)
print(f"{'Total':<15} {len(X):>8,} {'100.0':>12}% {y.mean()*100:>11.2f}%")
print()
print("Notice: fraud rate is ~2% in ALL three splits. Stratification works.")

In [ ]:
# ── Visualize the Three-Way Split ─────────────────────────────────────────────
# Stacked bar showing the composition of each split
# and a pie chart showing proportions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Stacked bar — legitimate vs fraud per split
split_names  = ['Train', 'Validation', 'Test']
legit_counts = [(ys==0).sum() for _, ys in splits.values()]
fraud_counts = [(ys==1).sum() for _, ys in splits.values()]

x_pos = np.arange(len(split_names))
axes[0].bar(x_pos, legit_counts, label='Legitimate', color='steelblue', alpha=0.85)
axes[0].bar(x_pos, fraud_counts, bottom=legit_counts, label='Fraud', color='tomato', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(split_names, fontsize=12)
axes[0].set_ylabel('Transaction Count', fontsize=11)
axes[0].set_title('Transaction Counts per Split\n(Fraud shown in red — scaled up for visibility)', fontsize=11)
axes[0].legend(fontsize=10)

# Add fraud rate labels
for i, (lc, fc) in enumerate(zip(legit_counts, fraud_counts)):
    axes[0].text(i, lc + fc + 30, f'{fc/(lc+fc)*100:.2f}% fraud',
                ha='center', fontsize=9, color='darkred')

# Right: Pie chart — split proportions
sizes = [len(X_train), len(X_val), len(X_test)]
labels_pie = [f'Train\n({len(X_train):,})', f'Val\n({len(X_val):,})', f'Test\n({len(X_test):,})']
colors_pie = ['steelblue', 'orange', 'tomato']
axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Three-Way Split Proportions\n(70% / 15% / 15%)', fontsize=11)

plt.suptitle('Credit Card Fraud Dataset — Three-Way Stratified Split', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4: Choosing Split Ratios — It Depends on Dataset Size

### The Core Principle

The validation and test sets need enough samples to give **reliable, stable estimates**. A test set with 10 samples is useless — the estimate has enormous variance. A test set with 10,000 samples gives a tight estimate.

**Rule of thumb:** Each of val and test should have at least **1,000 samples** (for most tasks). For rare events like fraud (2%), that means at least 50,000 samples total before a 1% test set gives you 1,000 tests.

| Dataset Size | Recommended Strategy | Notes |
|---|---|---|
| n < 1,000 | Cross-validation (no holdout val) | Too small for a stable holdout |
| 1K – 10K | 70/15/15 or 80/10/10 | Standard regime |
| 10K – 100K | 80/10/10 | Test set has 10K+ samples |
| 100K – 1M | 85/7.5/7.5 | Val/test still have 7.5K+ each |
| n > 1M | 95/2.5/2.5 or 98/1/1 | 1% of 1M = 10K — more than enough |

**Key insight:** As n grows, the *absolute size* of val/test matters more than the percentage. 1% of 10 million is 100K samples — far more than needed.

In [ ]:
# ── Ratio Experiment: How split ratio affects validation set variance ───────────
# We try three different train/val/test ratios
# For each, we run 30 different random seeds and measure variance in val accuracy
# Lower variance = more reliable validation estimate

ratios = [
    ('60/20/20', 0.20, 0.25),   # test=20%, val=25% of remaining 80%
    ('80/10/10', 0.10, 0.111),  # test=10%, val=11.1% of remaining 90%
    ('90/5/5',   0.05, 0.053),  # test=5%, val=5.3% of remaining 95%
]

n_seeds = 30
ratio_results = {}

for ratio_name, test_frac, val_frac in ratios:
    val_accs = []
    for seed in range(n_seeds):
        # Create three-way split
        X_tv, X_te, y_tv, y_te = train_test_split(
            X, y, test_size=test_frac, random_state=seed, stratify=y
        )
        X_tr, X_vl, y_tr, y_vl = train_test_split(
            X_tv, y_tv, test_size=val_frac, random_state=seed, stratify=y_tv
        )
        # Scale and train logistic regression
        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_tr)
        X_vl_sc = sc.transform(X_vl)
        lr = LogisticRegression(C=1.0, max_iter=500, random_state=0)
        lr.fit(X_tr_sc, y_tr)
        val_accs.append(accuracy_score(y_vl, lr.predict(X_vl_sc)))

    ratio_results[ratio_name] = val_accs
    print(f"{ratio_name}: Val size ≈ {int(len(X)*float(ratio_name.split('/')[1])/100):,} | "
          f"Mean={np.mean(val_accs):.4f} | Std={np.std(val_accs):.4f}")

print()
print("Higher std = less reliable estimate. Larger val set = lower std.")

In [ ]:
# ── Visualize ratio experiment: distribution of validation accuracy by split ratio
# Box plots and violin plots show the spread of accuracy across 30 random seeds
# A wider distribution means the validation estimate is noisier / less trustworthy

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Prepare data for plotting
plot_data = pd.DataFrame(ratio_results)
colors_map = {'60/20/20': 'steelblue', '80/10/10': 'orange', '90/5/5': 'tomato'}

# Left: Box plots
bp = axes[0].boxplot([ratio_results[r] for r in ratio_results],
                     labels=list(ratio_results.keys()),
                     patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors_map.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_xlabel('Train/Val/Test Ratio', fontsize=12)
axes[0].set_ylabel('Validation Accuracy (30 runs)', fontsize=12)
axes[0].set_title('Split Ratio vs. Validation Stability\n'
                  'Wider box = noisier estimate = less reliable', fontsize=11)

# Right: Variance bar chart
stds = [np.std(ratio_results[r]) for r in ratio_results]
bar_colors = list(colors_map.values())
bars = axes[1].bar(list(ratio_results.keys()), stds, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, std_val in zip(bars, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.00005,
                f'{std_val:.5f}', ha='center', va='bottom', fontsize=10)
axes[1].set_xlabel('Train/Val/Test Ratio', fontsize=12)
axes[1].set_ylabel('Std Dev of Validation Accuracy', fontsize=12)
axes[1].set_title('Validation Variance by Split Ratio\n'
                  'Smaller val set → higher variance → unreliable tuning', fontsize=11)

plt.suptitle('Effect of Split Ratio on Validation Reliability (30 random seeds)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 5: Random Seeds and Reproducibility

### Why Seeds Matter

`random_state=42` is not magic — the specific value doesn't matter. What matters is that:
1. You **set it** — so your split is reproducible across runs
2. You **don't change it** between runs to compare models
3. You're aware that **different seeds give different splits**, which give different accuracy numbers

### The Cherry-Picking Problem

If you run 20 different seeds and report the best one, you're not reporting generalization performance — you're reporting the luckiest draw. This is a form of test-set contamination through seed selection.

**Fraud analogy:** Imagine giving the certification exam 20 times and reporting only the best score. The reported score tells you nothing about the analyst's actual skill.

In [ ]:
# ── Seed Instability: Same model, different seeds → different test accuracy ─────
# We train the exact same logistic regression with the exact same hyperparameters
# The only thing that changes is the random seed used to create the split
# This shows how much luck influences a single reported accuracy number

n_seeds = 20
seed_results = []

for seed in range(n_seeds):
    # Create split with this seed
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y
    )
    # Scale: fit on train, apply to test
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    # Train logistic regression — same hyperparameters every time
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=0)
    lr.fit(X_tr_sc, y_tr)
    test_acc = accuracy_score(y_te, lr.predict(X_te_sc))
    seed_results.append({'seed': seed, 'test_accuracy': test_acc})

seed_df = pd.DataFrame(seed_results)

# ── Plot: accuracy distribution across seeds ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of accuracy per seed
axes[0].bar(seed_df['seed'], seed_df['test_accuracy'], color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axhline(seed_df['test_accuracy'].mean(), color='tomato', lw=2, linestyle='--',
                label=f'Mean: {seed_df["test_accuracy"].mean():.4f}')
axes[0].axhline(seed_df['test_accuracy'].max(), color='green', lw=1.5, linestyle=':',
                label=f'Best (cherry-picked): {seed_df["test_accuracy"].max():.4f}')
axes[0].axhline(seed_df['test_accuracy'].min(), color='orange', lw=1.5, linestyle=':',
                label=f'Worst: {seed_df["test_accuracy"].min():.4f}')
axes[0].set_xlabel('Random Seed', fontsize=11)
axes[0].set_ylabel('Test Accuracy', fontsize=11)
axes[0].set_title('Same Model, Different Splits:\nTest Accuracy Varies by Seed', fontsize=11)
axes[0].legend(fontsize=9)

# Right: histogram
axes[1].hist(seed_df['test_accuracy'], bins=10, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(seed_df['test_accuracy'].mean(), color='tomato', lw=2, linestyle='--',
                label=f'Mean ± Std\n{seed_df["test_accuracy"].mean():.4f} ± {seed_df["test_accuracy"].std():.4f}')
axes[1].set_xlabel('Test Accuracy', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Test Accuracy\nAcross 20 Random Splits', fontsize=11)
axes[1].legend(fontsize=10)

plt.suptitle('Random Seed Instability: Why You Should Report Mean ± Std, Not Best Run',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(f"If we cherry-pick the best seed: {seed_df['test_accuracy'].max():.4f}")
print(f"Honest mean across seeds:        {seed_df['test_accuracy'].mean():.4f}")
print(f"Inflation from cherry-picking:   {(seed_df['test_accuracy'].max() - seed_df['test_accuracy'].mean())*100:.2f} pp")

---
## Part 6: Test-Set Contamination — The Silent Killer of Honest Evaluation

### How Contamination Happens

Test-set contamination is subtle. It doesn't require malicious intent. It happens naturally when:

1. **Debugging on the test set:** "The model misclassifies foreign transactions — let me add a foreign_flag feature." (The decision was guided by test-set observations.)
2. **Feature scaling on full dataset:** Fit StandardScaler on all 10,000 samples including test set — the test set's statistics leak into the scaler.
3. **Reporting the best of N models:** Train 20 models, report the one with highest test accuracy — you've implicitly used test performance to select a model.
4. **Multiple test evaluations:** Every time you look at test performance and make any decision, some information leaks.

### The Multiple Comparisons Problem

If you compare 20 models on the same test set and pick the best, the expected test accuracy of the winner is higher than its true generalization accuracy. By sheer luck, one model will get a favorable test split.

$$P(\text{at least one model looks good by luck}) = 1 - (1-\alpha)^{20}$$

With α=0.05 and 20 models: $1 - 0.95^{20} \approx 64\%$ chance of false discovery.

In [ ]:
# ── Test-Set Contamination Demo: Simulated "peeking" loop ─────────────────────
# We simulate an analyst who repeatedly looks at test performance
# and adds random features whenever test accuracy isn't good enough
# This is the contamination loop: test score → decision → retrain → repeat
# Watch the test accuracy inflate far beyond what the val set shows

np.random.seed(42)

# Fixed split for this experiment
X_tv, X_te, y_tv, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_tv, y_tv, test_size=0.20, random_state=42, stratify=y_tv)

contamination_log = []

# Start with the 10 original features
X_tr_aug, X_vl_aug, X_te_aug = X_tr.copy(), X_vl.copy(), X_te.copy()

for iteration in range(15):
    # Scale and train
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_aug)
    X_vl_sc = sc.transform(X_vl_aug)
    X_te_sc = sc.transform(X_te_aug)

    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=0)
    lr.fit(X_tr_sc, y_tr)

    val_acc  = accuracy_score(y_vl, lr.predict(X_vl_sc))
    test_acc = accuracy_score(y_te, lr.predict(X_te_sc))  # ← PEEKING!

    contamination_log.append({
        'iteration': iteration,
        'n_features': X_tr_aug.shape[1],
        'val_accuracy': val_acc,
        'test_accuracy': test_acc
    })

    # "Fix" model by adding a random noise feature (guided by test observation)
    # In practice this might be: "I noticed test fails on high-amount txns,
    # so I'll engineer a new 'high_amount' feature"
    noise_feature = np.random.randn(len(X_tr_aug), 1)  # pure noise — no signal
    X_tr_aug = np.hstack([X_tr_aug, np.random.randn(len(X_tr_aug), 1)])
    X_vl_aug = np.hstack([X_vl_aug, np.random.randn(len(X_vl_aug), 1)])
    X_te_aug = np.hstack([X_te_aug, np.random.randn(len(X_te_aug), 1)])

contamination_df = pd.DataFrame(contamination_log)
print(contamination_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ── Visualize contamination: val accuracy stays flat, test accuracy rises ──────
# The val accuracy (honest estimate) stays stable — we're adding noise features
# The test accuracy appears to improve because the loop is guided by test performance
# This is accuracy inflation caused by implicit test-set leakage

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy over iterations
axes[0].plot(contamination_df['iteration'], contamination_df['val_accuracy'],
             'o-', color='steelblue', lw=2, label='Validation Accuracy (honest)')
axes[0].plot(contamination_df['iteration'], contamination_df['test_accuracy'],
             's-', color='tomato', lw=2, label='Test Accuracy (contaminated)')
axes[0].fill_between(contamination_df['iteration'],
                     contamination_df['val_accuracy'],
                     contamination_df['test_accuracy'],
                     alpha=0.15, color='red', label='Contamination gap')
axes[0].set_xlabel('Iteration (each adds one noise feature after peeking at test)', fontsize=10)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Test-Set Contamination:\nPeeking Inflates Test Accuracy', fontsize=11)
axes[0].legend(fontsize=10)

# Right: contamination gap bar chart
gap = contamination_df['test_accuracy'] - contamination_df['val_accuracy']
bar_colors = ['tomato' if g > 0 else 'steelblue' for g in gap]
axes[1].bar(contamination_df['iteration'], gap, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Test Acc − Val Acc (Contamination Gap)', fontsize=11)
axes[1].set_title('Growing Gap Between Test and Validation:\nEvidence of Contamination', fontsize=11)

plt.suptitle('Test-Set Contamination: Why You Must NEVER Use Test Results to Guide Development',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

initial_gap = contamination_df.iloc[0]['test_accuracy'] - contamination_df.iloc[0]['val_accuracy']
final_gap   = contamination_df.iloc[-1]['test_accuracy'] - contamination_df.iloc[-1]['val_accuracy']
print(f"Gap at start: {initial_gap:+.4f}")
print(f"Gap after 15 iterations of peeking: {final_gap:+.4f}")
print(f"Contamination inflated apparent test performance by {(final_gap - initial_gap)*100:.2f} pp")

---
## Part 7: Common Mistakes in Holdout Validation

Here is a catalogue of the most frequent errors, with the fraud detection context:

### Mistake 1: Fitting the Scaler on All Data (Including Test)

```python
# WRONG: Test set statistics leak into the scaler
scaler.fit(X_all)          # sees test set!
X_train_sc = scaler.transform(X_train)
X_test_sc  = scaler.transform(X_test)

# CORRECT: Fit only on training data
scaler.fit(X_train)        # never sees test
X_train_sc = scaler.transform(X_train)
X_test_sc  = scaler.transform(X_test)   # apply same transform
```

### Mistake 2: Reporting Best of N Test Evaluations

```python
# WRONG: Cherry-picking best test result across 20 models
best_acc = max(accuracy_score(y_test, model.predict(X_test)) for model in models)
print(f"Our model achieves {best_acc:.3f}")  # biased upward!

# CORRECT: Use val set to select model, then evaluate once on test
best_model = max(models, key=lambda m: accuracy_score(y_val, m.predict(X_val)))
final_acc = accuracy_score(y_test, best_model.predict(X_test))  # one time!
```

### Mistake 3: Skipping Stratification on Imbalanced Data

```python
# WRONG: Random split might miss rare fraud cases in test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# CORRECT: Always stratify when class imbalance exists
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
```

### Mistake 4: No Random Seed

```python
# WRONG: Each run gives a different split → results not reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# CORRECT: Set random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

In [ ]:
# ── Data Leakage Demo: Scaling leakage ────────────────────────────────────────
# Demonstrates that fitting scaler on all data causes subtle but real leakage
# The difference is usually small but matters for edge-case decisions
# In real pipelines, leakage from imputation or encoding can be larger

# Create fresh split for this experiment
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# WRONG approach: fit scaler on all data
sc_wrong = StandardScaler()
sc_wrong.fit(X)           # LEAKAGE — test set influences the scaler
X_tr_wrong = sc_wrong.transform(X_tr_leak)
X_te_wrong = sc_wrong.transform(X_te_leak)

# CORRECT approach: fit scaler only on training data
sc_right = StandardScaler()
sc_right.fit(X_tr_leak)   # NO LEAKAGE — test set is invisible
X_tr_right = sc_right.transform(X_tr_leak)
X_te_right = sc_right.transform(X_te_leak)

# Train and evaluate both
results_leak = {}
for approach, Xtr, Xte in [
    ('Wrong (full-data scaling)', X_tr_wrong, X_te_wrong),
    ('Correct (train-only scaling)', X_tr_right, X_te_right)
]:
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    lr.fit(Xtr, y_tr_leak)
    acc = accuracy_score(y_te_leak, lr.predict(Xte))
    results_leak[approach] = acc
    print(f"{approach}: Test Accuracy = {acc:.5f}")

# Show the difference in scaler parameters (mean of feature 0)
print(f"\nScaler mean for 'txn_amount' (feature 0):")
print(f"  Fitted on all data:   {sc_wrong.mean_[0]:.6f}")
print(f"  Fitted on train only: {sc_right.mean_[0]:.6f}")
print(f"  Difference:           {abs(sc_wrong.mean_[0] - sc_right.mean_[0]):.6f}")
print()
print("The difference is small here because scaling leakage is mild.")
print("With imputation leakage (e.g., filling missing values with test-set mean),")
print("the inflated performance can be much larger.")

In [ ]:
# ── Split Strategy Recommendation: Dataset size → best approach ────────────────
# This serves as a practical reference guide for choosing split strategy
# Numbers are guidelines — always adapt to your specific problem's rarity of events

print("=" * 80)
print("SPLIT STRATEGY REFERENCE: Dataset Size → Recommended Approach")
print("=" * 80)
print()

recommendations = [
    ("< 500",    "5-fold or 10-fold CV",            "Too small for holdout. Every sample needed for training."),
    ("500–1K",   "Stratified 5-fold CV",            "Holdout val set would be too small (< 50-100 samples)."),
    ("1K–5K",    "70/15/15 stratified",             "Minimum viable holdout. ~150 samples in val/test."),
    ("5K–50K",   "80/10/10 stratified",             "Standard regime. 500–5K samples in val/test."),
    ("50K–500K", "85/7.5/7.5 stratified",          "Large enough. 3.5K–37K samples in val/test."),
    (">500K",    "95/2.5/2.5 or 98/1/1 stratified", "Even 1% gives 5K+ samples — more than sufficient."),
]

print(f"{'Dataset Size':<15} {'Strategy':<30} {'Rationale'}")
print("-" * 80)
for size, strategy, rationale in recommendations:
    print(f"{size:<15} {strategy:<30} {rationale}")

print()
print("=" * 80)
print("ADDITIONAL RULES:")
print("  1. ALWAYS stratify when the minority class is < 10% of the data.")
print("  2. ALWAYS set random_state for reproducibility.")
print("  3. NEVER fit preprocessing (scaler, imputer, encoder) on test data.")
print("  4. NEVER look at test performance until all development is complete.")
print("  5. Val and test sets should each have ≥ 500 minority-class samples.")
print("=" * 80)

In [ ]:
# ── Final Visualization: The complete holdout validation workflow ───────────────
# A flowchart-style bar diagram showing the correct order of operations
# This is the mental model you should carry with you

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 10)
ax.set_ylim(-1, 8)
ax.axis('off')

# Define boxes in the workflow
steps = [
    (5.0, 7.0, "RAW DATASET\n(10,000 fraud transactions)",     'lightgray',   'black'),
    (5.0, 5.5, "Lock away TEST SET (15%)\nDo not touch until Step 6", '#ffcccc', 'darkred'),
    (5.0, 4.0, "TRAIN SET (70%)\nFit model parameters",         '#cce5ff', 'darkblue'),
    (5.0, 2.5, "VALIDATION SET (15%)\nTune hyperparameters, compare models", '#fff3cd', 'darkorange'),
    (5.0, 1.0, "Iterate train → val → adjust\n(as many times as needed)", '#d4edda', 'darkgreen'),
    (5.0,-0.5, "Final TEST SET evaluation\n(exactly once — report this number)", '#ffcccc', 'darkred'),
]

for x, y_pos, text, bgcolor, textcolor in steps:
    ax.add_patch(plt.Rectangle((x-4, y_pos-0.55), 8, 1.0,
                               facecolor=bgcolor, edgecolor='gray',
                               linewidth=1.5, zorder=2))
    ax.text(x, y_pos, text, ha='center', va='center',
            fontsize=11, color=textcolor, fontweight='bold', zorder=3)

# Arrows between steps
arrow_props = dict(arrowstyle='->', color='gray', lw=2)
for y_start, y_end in [(6.5, 6.1), (4.95, 4.61), (3.45, 3.11), (1.95, 1.61), (0.45, 0.11)]:
    ax.annotate('', xy=(5.0, y_end), xytext=(5.0, y_start),
                arrowprops=arrow_props)

ax.set_title('The Correct Holdout Validation Workflow\n'
             'Test set = one-time use instrument; validation set = compass during development',
             fontsize=13, pad=20)
plt.tight_layout()
plt.show()

print("Workflow Summary:")
print("  1. Split: create train/val/test (stratified, fixed seed)")
print("  2. Lock away test set — don't look at it")
print("  3. Train models on training set")
print("  4. Evaluate on validation set — tune, improve, iterate")
print("  5. When satisfied, run evaluation on test set — ONCE")
print("  6. Report that number. Do not go back and adjust.")

---
## Part 8: Summary

### Key Takeaways

| Concept | One-Line Summary |
|---|---|
| **Why three sets** | Train fits; val tunes; test gives final honest estimate |
| **Test set** | One-time instrument — like a final exam, no retakes |
| **Stratification** | Essential for imbalanced data; preserves class proportions |
| **Random seed** | Always set it — different seeds, different results |
| **Contamination** | Every time test score guides a decision, it inflates accuracy |
| **Split ratio** | Depends on n — larger datasets allow smaller % for val/test |
| **Scaling rule** | Fit preprocessing on train only; apply to val/test |

### The Cardinal Rules

> 1. **The test set is touched once.** If you look at test performance and adjust anything, you've contaminated it.
> 2. **Preprocessing is fitted on training data only.** Scalers, imputers, encoders — all fitted on train, applied to val/test.
> 3. **Stratify when class imbalance exists.** Two lines of code; prevents silent evaluation failures.

### What's Next

In Notebook 3, we cover **Cross-Validation** — the solution when your dataset is too small for a reliable holdout validation set, and the gold standard for hyperparameter tuning.

---
## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You train 5 different fraud detection models (logistic regression, decision tree, random forest, SVM, k-NN) and report the one with the highest test set performance. Why is this accuracy estimate optimistically biased?

<details>
<summary>Click to reveal answer</summary>

**You have implicitly used the test set as a model selection criterion — this is test-set contamination through the multiple comparisons problem.**

When you evaluate 5 models on the same test set and pick the winner, the winning model's test accuracy is higher than its true generalization accuracy. This happens for two reasons:

1. **Random chance:** By luck, one model happens to be favorably matched to the particular test examples drawn. A different test set would pick a different winner.
2. **Implicit optimization:** Your choice of "best model" was driven by test set performance. You have now implicitly optimized a decision using the test set — it is no longer a neutral evaluator.

**The correct procedure:** Use the validation set to select the best model. Then evaluate that single chosen model on the test set. The test score is reported without knowledge of how other models performed on the test set.

</details>

---

**Question 2:** You have 500 training samples and 50 fraud cases (10% fraud rate — still imbalanced but less extreme). Should you use a holdout validation set or cross-validation? Justify your answer.

<details>
<summary>Click to reveal answer</summary>

**Use cross-validation (specifically stratified k-fold). Do not use a holdout validation set.**

With 500 samples and a 70/15/15 split:
- Training set: 350 samples, ~35 fraud cases
- Validation set: 75 samples, ~7 fraud cases
- Test set: 75 samples, ~7 fraud cases

A validation set of 75 samples with only 7 fraud cases is far too small to give a reliable estimate. Accuracy on 7 fraud cases changes by ~14% with every single misclassification — this is too noisy to guide hyperparameter decisions.

With 5-fold stratified cross-validation:
- Each validation fold has 100 samples, 10 fraud cases
- You get 5 estimates and average them → much lower variance
- All 500 samples contribute to training across folds

Rule: when any split would have fewer than ~50 minority-class samples, use cross-validation instead of holdout.

</details>

---

**Question 3:** A colleague says: "I checked the test set and the model is misclassifying high-value foreign transactions, so I engineered a new feature combining amount and foreign_flag, retrained, and now we get 96% test accuracy (up from 93%)." What went wrong, and what should they have done instead?

<details>
<summary>Click to reveal answer</summary>

**The colleague contaminated the test set.** The 96% figure is no longer a valid measure of generalization.

**What went wrong:**
The decision to engineer the `amount × foreign_flag` feature was directly guided by observing errors in the test set. This means:
1. The new feature was specifically designed to fix patterns observed in the test set
2. The model is now partially tuned to the test set's specific examples
3. The 96% accuracy is inflated — it reflects some memorization of the test set's failure modes

**What should have been done:**
1. **Error analysis on the validation set** — look at misclassified examples in the validation set to identify patterns
2. Engineer the new feature based on those validation-set insights
3. Retrain on the training set
4. Check performance on the validation set (not test set)
5. Iterate steps 2–4 until satisfied
6. **Only then**, as the final step, evaluate on the test set once and report that result

The test set should be a black box during development. It answers one question at the very end: "How does this model perform on data it has never influenced?"

</details>